In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [1]:
import json
from pathlib import Path

INPUT_FILE = Path("audit/new_documents.json")

with open(INPUT_FILE, "r", encoding="utf-8") as file:
    new_documents = json.load(file)

print(f"New documents loaded: {len(new_documents)}")

for doc in new_documents:
    print(
        doc["title"],
        "|",
        doc["batch_id"],
        "|",
        len(doc["text"]),
        "characters"
    )

New documents loaded: 3
Apache Airflow | batch_4 | 2426 characters
Apache Kafka | batch_4 | 2303 characters
Data Lakehouse Architecture | batch_4 | 2372 characters


In [2]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

for doc in new_documents:
    token_count = len(encoding.encode(doc["text"]))

    print(
        doc["title"],
        "| tokens:",
        token_count
    )

Apache Airflow | tokens: 462
Apache Kafka | tokens: 429
Data Lakehouse Architecture | tokens: 419


In [3]:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

chunks = []

for doc in new_documents:
    tokens = encoding.encode(doc["text"])
    start = 0
    chunk_number = 1

    while start < len(tokens):
        end = start + CHUNK_SIZE
        chunk_tokens = tokens[start:end]

        chunk_text = encoding.decode(chunk_tokens)

        chunks.append({
            "chunk_id": f"{doc['document_id']}_chunk_{chunk_number}",
            "document_id": doc["document_id"],
            "title": doc["title"],
            "source_file": doc["source_file"],
            "batch_id": doc["batch_id"],
            "content_hash": doc["content_hash"],
            "text": chunk_text
        })

        chunk_number += 1
        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Documents processed: {len(new_documents)}")
print(f"Total chunks created: {len(chunks)}")

for doc in new_documents:
    count = sum(
        1 for chunk in chunks
        if chunk["document_id"] == doc["document_id"]
    )
    print(doc["title"], "| chunks:", count)

Documents processed: 3
Total chunks created: 6
Apache Airflow | chunks: 2
Apache Kafka | chunks: 2
Data Lakehouse Architecture | chunks: 2


In [4]:
if chunks:
    sample_chunk = chunks[0]

    print("Chunk ID:", sample_chunk["chunk_id"])
    print("Document ID:", sample_chunk["document_id"])
    print("Title:", sample_chunk["title"])
    print("Source File:", sample_chunk["source_file"])
    print("Batch:", sample_chunk["batch_id"])

    print("\nChunk Text:\n")
    print(sample_chunk["text"])
else:
    print("No new chunks available to preview.")

Chunk ID: DOC11_chunk_1
Document ID: DOC11
Title: Apache Airflow
Source File: apache_airflow.pdf
Batch: batch_4

Chunk Text:

Apache Airflow

Airflow Overview
Apache Airflow is a workflow orchestration platform used to define, schedule, and monitor data
workflows. A workflow is represented as a Directed Acyclic Graph, or DAG, where tasks describe
individual units of work and dependencies describe the order in which those tasks should run. Airflow is
commonly used in data engineering to coordinate ingestion, transformation, validation, and downstream
processing without embedding the business logic inside the scheduler itself.
DAGs, Tasks, and Dependencies
A DAG defines the structure of a workflow. Tasks may run Python functions, execute SQL, trigger
external jobs, or call other systems through operators and hooks. Dependencies such as task_a >>
task_b make the execution order explicit. This is useful for pipelines where a transformation must wait
for ingestion to finish, or a validation

In [5]:
def get_category(title):
    title_lower = title.lower()

    if "rag" in title_lower:
        return "RAG"
    elif "data factory" in title_lower:
        return "Azure Data Factory"
    elif "databricks" in title_lower:
        return "Azure Databricks"
    elif "spark" in title_lower or "pyspark" in title_lower:
        return "Spark"
    elif "mysql" in title_lower:
        return "MySQL"
    elif "data modeling" in title_lower:
        return "Data Modeling"
    elif "python" in title_lower:
        return "Python"
    else:
        return "Data Engineering"


for chunk in chunks:
    chunk["category"] = get_category(chunk["title"])

print("Categories added.")

for chunk in chunks[:5]:
    print(
        chunk["chunk_id"],
        "|",
        chunk["title"],
        "|",
        chunk["category"]
    )

Categories added.
DOC11_chunk_1 | Apache Airflow | Data Engineering
DOC11_chunk_2 | Apache Airflow | Data Engineering
DOC12_chunk_1 | Apache Kafka | Data Engineering
DOC12_chunk_2 | Apache Kafka | Data Engineering
DOC13_chunk_1 | Data Lakehouse Architecture | Data Engineering


In [6]:
import chromadb

CHROMA_PATH = "vector_store"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="technical_docs"
)

print("ChromaDB collection ready.")
print("Existing records:", collection.count())

ChromaDB collection ready.
Existing records: 31


In [7]:
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_metadatas = [
    {
        "document_id": chunk["document_id"],
        "title": chunk["title"],
        "source_file": chunk["source_file"],
        "batch_id": chunk["batch_id"],
        "category": chunk["category"]
    }
    for chunk in chunks
]

print("Prepared chunks for embedding:", len(chunk_ids))
print("Prepared metadata records:", len(chunk_metadatas))

Prepared chunks for embedding: 6
Prepared metadata records: 6


In [8]:
CHUNKS_FILE = Path("audit/new_chunks.json")

with open(CHUNKS_FILE, "w", encoding="utf-8") as file:
    json.dump(chunks, file, indent=2)

print(f"Saved {len(chunks)} chunks for embedding.")

Saved 6 chunks for embedding.


In [9]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as file:
    saved_chunks = json.load(file)

print("Saved chunks:", len(saved_chunks))

if saved_chunks:
    print("First chunk ID:", saved_chunks[0]["chunk_id"])
    print("First chunk title:", saved_chunks[0]["title"])
    print("First chunk category:", saved_chunks[0]["category"])
else:
    print("No saved chunks available to preview.")

Saved chunks: 6
First chunk ID: DOC11_chunk_1
First chunk title: Apache Airflow
First chunk category: Data Engineering


In [10]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EMBEDDING_MODEL = "text-embedding-3-large"

if chunk_texts:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=chunk_texts
    )

    embeddings = [
        item.embedding
        for item in response.data
    ]

    print("Embeddings generated:", len(embeddings))
    print("Embedding dimension:", len(embeddings[0]))
else:
    embeddings = []
    print("No new chunks found. Embedding generation skipped.")

Embeddings generated: 6
Embedding dimension: 3072


In [11]:
if chunk_ids:
    collection.add(
        ids=chunk_ids,
        embeddings=embeddings,
        documents=chunk_texts,
        metadatas=chunk_metadatas
    )

    print("Chunks stored in ChromaDB:", len(chunk_ids))
    print("Total records in ChromaDB:", collection.count())
else:
    print("No new chunks to store in ChromaDB.")
    print("Total records in ChromaDB:", collection.count())

Chunks stored in ChromaDB: 6
Total records in ChromaDB: 37


In [12]:
PROCESSED_FILE = Path("audit/processed_documents.json")

processed_documents = []

if PROCESSED_FILE.exists():
    with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
        processed_documents = json.load(file)

processed_hashes = {
    doc["content_hash"]
    for doc in processed_documents
}

for doc in new_documents:
    if doc["content_hash"] not in processed_hashes:
        processed_documents.append({
            "document_id": doc["document_id"],
            "source_file": doc["source_file"],
            "batch_id": doc["batch_id"],
            "content_hash": doc["content_hash"]
        })

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_documents, file, indent=2)

print("Total processed documents:", len(processed_documents))

Total processed documents: 13
